# Pandas para contadores — SolucionesEjecuta las celdas en orden: cada ejercicio usa las variables del anterior.Cada solución trae comentarios de lo que hay que mirar como contador, no solo como programador.---

## Ejercicio 1 — Leer los archivosPrimer contacto con `pandas`. Un **DataFrame** es una tabla; una **Series** es una columna.Carga los cuatro archivos de la carpeta `datos/` y muestra, de cada uno, cuántas filas y columnas tiene.> **Ojo profesional:** `codigo_puc` y los NIT son *códigos*, no números. Si dejas que pandas los lea como enteros pierdes los ceros a la izquierda y no podrás cruzarlos después. Usa `dtype={"codigo_puc": str}`.

In [ ]:
import pandas as pdRUTA = "datos/"aux = pd.read_csv(RUTA + "libro_auxiliar.csv", dtype={"codigo_puc": str, "nit_tercero": str})bp  = pd.read_csv(RUTA + "balance_prueba.csv", dtype={"codigo_puc": str})ext = pd.read_csv(RUTA + "extracto_bancario.csv")fac = pd.read_csv(RUTA + "facturas_proveedores.csv", dtype={"nit_proveedor": str})for nombre, df in [("libro auxiliar", aux), ("balance de prueba", bp),                   ("extracto bancario", ext), ("facturas", fac)]:    print(f"{nombre:20s} filas={df.shape[0]:5d}  columnas={df.shape[1]}")aux.head()

## Ejercicio 2 — Radiografía del libro auxiliarAntes de calcular nada hay que saber **qué tipo de dato** tiene cada columna. Usa:- `.info()` — tipos y nulos- `.dtypes` — solo tipos- `.describe()` — estadísticos de las columnas numéricas- `.columns` — nombres de columnasAdemás convierte `fecha` a tipo fecha real con `pd.to_datetime`, porque leída del CSV llega como texto.

In [ ]:
aux.info()print()aux["fecha"] = pd.to_datetime(aux["fecha"])ext["fecha"] = pd.to_datetime(ext["fecha"])print("Rango del periodo:", aux["fecha"].min().date(), "a", aux["fecha"].max().date())print("Comprobantes distintos:", aux["comprobante"].nunique())print("Cuentas distintas:", aux["codigo_puc"].nunique())aux[["debito", "credito"]].describe()

## Ejercicio 3 — Partida doble: ¿cuadra el auxiliar?La primera validación de cualquier archivo que te entreguen: **suma de débitos = suma de créditos**.Calcula ambas sumas, su diferencia, y escribe una condición que imprima `CUADRA` o `NO CUADRA`.> Redondea a 2 decimales antes de comparar: los flotantes arrastran error binario y `0.1 + 0.2 != 0.3` en Python.

In [ ]:
total_debitos = aux["debito"].sum()total_creditos = aux["credito"].sum()diferencia = round(total_debitos - total_creditos, 2)print(f"Total debitos : {total_debitos:>18,.2f}")print(f"Total creditos: {total_creditos:>18,.2f}")print(f"Diferencia    : {diferencia:>18,.2f}")print("CUADRA" if diferencia == 0 else "NO CUADRA - revisar")

## Ejercicio 4 — Seleccionar y filtrarTres formas de acceder a los datos:| Qué quieres | Cómo ||---|---|| una columna | `df["debito"]` || varias columnas | `df[["fecha", "debito"]]` || filas por condición | `df[df["codigo_puc"] == "111005"]` || fila/columna por etiqueta | `df.loc[fila, "columna"]` || fila/columna por posición | `df.iloc[0, 3]` |Extrae todos los movimientos de la cuenta **111005 (banco)**, cuenta cuántos son y calcula el movimiento neto (débitos − créditos).

In [ ]:
banco = aux[aux["codigo_puc"] == "111005"]print("Movimientos de banco:", len(banco))print(f"Debitos : {banco['debito'].sum():,.2f}")print(f"Creditos: {banco['credito'].sum():,.2f}")print(f"Neto    : {banco['debito'].sum() - banco['credito'].sum():,.2f}")banco.head()

## Ejercicio 5 — Filtros compuestosSe combinan con `&` (y), `|` (o), `~` (no), y **cada condición va entre paréntesis**.Encuentra los movimientos que cumplan las tres cosas:1. son de compras — el `comprobante` empieza por `FC`2. son de la cuenta de inventario `143530`3. el débito supera $3.000.000Pistas: `.str.startswith("FC")`, y para el mes `aux["fecha"].dt.month`.

In [ ]:
compras_grandes = aux[    (aux["comprobante"].str.startswith("FC"))    & (aux["codigo_puc"] == "143530")    & (aux["debito"] > 3_000_000)]print("Compras de inventario > 3.000.000:", len(compras_grandes))print(f"Valor total: {compras_grandes['debito'].sum():,.2f}")compras_grandes[["fecha", "comprobante", "nombre_tercero", "debito"]].head(10)

## Ejercicio 6 — Limpieza de las facturas de proveedoresEl archivo `facturas_proveedores.csv` viene sucio, como en la vida real. Corrige:1. **Espacios y mayúsculas** en `proveedor` → `.str.strip()` y `.str.upper()`2. **Filas duplicadas** → `.duplicated().sum()` y `.drop_duplicates()`3. **Nulos** en `estado` → `.isna().sum()` y `.fillna("PENDIENTE")`4. **Fechas** en formato `dd/mm/yyyy` → `pd.to_datetime(..., format="%d/%m/%Y")`Trabaja sobre una copia (`fac.copy()`) para no dañar el original.

In [ ]:
fac_limpio = fac.copy()print("Antes  -> filas:", len(fac_limpio),      "| duplicados:", fac_limpio.duplicated().sum(),      "| estados nulos:", fac_limpio["estado"].isna().sum())fac_limpio["proveedor"] = fac_limpio["proveedor"].str.strip().str.upper()fac_limpio = fac_limpio.drop_duplicates().reset_index(drop=True)fac_limpio["estado"] = fac_limpio["estado"].fillna("PENDIENTE")fac_limpio["fecha_factura"] = pd.to_datetime(fac_limpio["fecha_factura"], format="%d/%m/%Y")print("Despues-> filas:", len(fac_limpio),      "| duplicados:", fac_limpio.duplicated().sum(),      "| estados nulos:", fac_limpio["estado"].isna().sum())print("Proveedores unicos:", fac_limpio["proveedor"].nunique())fac_limpio.head()

## Ejercicio 7 — Recalcular las retencionesNunca confíes en las retenciones que trae el archivo: recalcúlalas.Sobre `fac_limpio` crea:- `rf_calc` = 2,5 % de la base gravable (compras generales)- `ica_calc` = 9,66 × 1.000 de la base gravable- `dif_rf` y `dif_ica` = diferencia contra lo registrado- una columna `alerta` que diga `OK` o `REVISAR` cuando la diferencia absoluta supere $1Luego muestra solo las filas con alerta.

In [ ]:
fac_limpio["rf_calc"] = (fac_limpio["base_gravable"] * 0.025).round(2)fac_limpio["ica_calc"] = (fac_limpio["base_gravable"] * 0.00966).round(2)fac_limpio["dif_rf"] = (fac_limpio["retefuente"] - fac_limpio["rf_calc"]).round(2)fac_limpio["dif_ica"] = (fac_limpio["reteica"] - fac_limpio["ica_calc"]).round(2)fac_limpio["alerta"] = "OK"fac_limpio.loc[    (fac_limpio["dif_rf"].abs() > 1) | (fac_limpio["dif_ica"].abs() > 1), "alerta"] = "REVISAR"print(fac_limpio["alerta"].value_counts())fac_limpio[["numero_factura", "proveedor", "base_gravable",            "retefuente", "rf_calc", "dif_rf",            "reteica", "ica_calc", "dif_ica", "alerta"]].head(10)

## Ejercicio 8 — groupby: ventas por mes`groupby` es el equivalente a una tabla dinámica. La receta es siempre la misma:```pythondf.groupby("columna_que_agrupa")["columna_a_sumar"].sum()```Crea una columna `mes` (`aux["fecha"].dt.to_period("M")`) y calcula las **ventas por mes**: créditos de la cuenta `413595`.

In [ ]:
aux["mes"] = aux["fecha"].dt.to_period("M")ventas_mes = (aux[aux["codigo_puc"] == "413595"]              .groupby("mes")["credito"].sum()              .rename("ventas"))print(ventas_mes.to_string())print(f"\nTotal del trimestre: {ventas_mes.sum():,.2f}")print(f"Variacion feb vs ene: {(ventas_mes.iloc[1] / ventas_mes.iloc[0] - 1) * 100:,.2f} %")

## Ejercicio 9 — Top de clientes y de proveedoresCon `groupby` + `sort_values` + `head` sacas cualquier ranking.1. Top 5 **clientes** por ventas (créditos de `413595` agrupados por `nombre_tercero`)2. Top 5 **proveedores** por compras (débitos de `143530` en comprobantes `FC`)3. Para el top de clientes agrega la columna `participacion` en % sobre el total.

In [ ]:
top_clientes = (aux[aux["codigo_puc"] == "413595"]                .groupby("nombre_tercero")["credito"].sum()                .sort_values(ascending=False)                .head(5)                .to_frame("ventas"))top_clientes["participacion_%"] = (    top_clientes["ventas"] / aux.loc[aux["codigo_puc"] == "413595", "credito"].sum() * 100).round(2)top_prov = (aux[(aux["codigo_puc"] == "143530") & (aux["comprobante"].str.startswith("FC"))]            .groupby("nombre_tercero")["debito"].sum()            .sort_values(ascending=False)            .head(5)            .to_frame("compras"))print("TOP 5 CLIENTES\n", top_clientes, "\n")print("TOP 5 PROVEEDORES\n", top_prov)

## Ejercicio 10 — pivot_table: gastos por centro de costo y mes`pivot_table` arma la tabla dinámica con filas, columnas y totales:```pythondf.pivot_table(index=..., columns=..., values=..., aggfunc="sum", margins=True)```Filtra las cuentas de gasto (las que empiezan por `5` — usa `.str.startswith("5")`) y arma la tabla con centro de costo en filas, mes en columnas y débitos como valor, con totales.

In [ ]:
gastos = aux[aux["codigo_puc"].str.startswith("5")]tabla = gastos.pivot_table(    index="centro_costo",    columns="mes",    values="debito",    aggfunc="sum",    margins=True,    margins_name="TOTAL",).fillna(0).round(2)tabla

## Ejercicio 11 — Reconstruir el balance de prueba desde el auxiliarEste es el cruce que hace un auditor: el balance que te entregan **debe** poder reconstruirse desde el auxiliar.1. Agrupa el auxiliar por `codigo_puc` sumando débitos y créditos2. Cruza (`merge`) ese resultado con `bp` por `codigo_puc`3. Calcula `dif_debitos` y `dif_creditos`4. Muestra las cuentas donde la diferencia no sea ceroSobre `merge`: `how="outer"` te deja ver también las cuentas que están en un archivo y no en el otro — justo lo que quieres detectar.

In [ ]:
mov = (aux.groupby("codigo_puc")[["debito", "credito"]].sum()       .rename(columns={"debito": "debitos_aux", "credito": "creditos_aux"})       .reset_index())cruce = bp.merge(mov, on="codigo_puc", how="outer", indicator=True)cruce[["debitos", "creditos", "debitos_aux", "creditos_aux"]] = \    cruce[["debitos", "creditos", "debitos_aux", "creditos_aux"]].fillna(0)cruce["dif_debitos"] = (cruce["debitos"] - cruce["debitos_aux"]).round(2)cruce["dif_creditos"] = (cruce["creditos"] - cruce["creditos_aux"]).round(2)descuadres = cruce[(cruce["dif_debitos"] != 0) | (cruce["dif_creditos"] != 0)]print("Cuentas con diferencia:", len(descuadres))print(cruce["_merge"].value_counts().to_string())descuadres[["codigo_puc", "nombre_cuenta", "dif_debitos", "dif_creditos"]]

## Ejercicio 12 — La ecuación contableLa clase de la cuenta es el primer dígito del PUC: `1` activo, `2` pasivo, `3` patrimonio, `4` ingreso, `5` gasto, `6` costo.1. Crea en `bp` la columna `clase` con `.str[0]`2. Suma `saldo_final` por clase (recuerda: en este archivo los saldos crédito vienen en negativo)3. Verifica que **Activo = Pasivo + Patrimonio + Resultado del ejercicio**Pista: si los saldos crédito son negativos, la suma de *todos* los saldos finales debe dar cero.

In [ ]:
bp["clase"] = bp["codigo_puc"].str[0]nombres = {"1": "ACTIVO", "2": "PASIVO", "3": "PATRIMONIO",           "4": "INGRESOS", "5": "GASTOS", "6": "COSTOS"}bp["clase_nombre"] = bp["clase"].map(nombres)resumen = bp.groupby("clase_nombre")["saldo_final"].sum().round(2)print(resumen.to_string(), "\n")activo = resumen.get("ACTIVO", 0)pasivo = -resumen.get("PASIVO", 0)patrimonio = -resumen.get("PATRIMONIO", 0)resultado = -(resumen.get("INGRESOS", 0) + resumen.get("GASTOS", 0) + resumen.get("COSTOS", 0))print(f"Activo                     : {activo:>18,.2f}")print(f"Pasivo                     : {pasivo:>18,.2f}")print(f"Patrimonio                 : {patrimonio:>18,.2f}")print(f"Resultado del ejercicio    : {resultado:>18,.2f}")print(f"Pasivo + Patrimonio + Rdo  : {pasivo + patrimonio + resultado:>18,.2f}")print("ECUACION OK" if round(activo - (pasivo + patrimonio + resultado), 2) == 0 else "DESCUADRE")

## Ejercicio 13 — Estado de resultados del trimestreArma el P&G con las clases 4, 6 y 5:| Renglón | Fórmula ||---|---|| Ingresos operacionales | créditos 41 − débitos 41 || Costo de ventas | clase 6 || **Utilidad bruta** | Ingresos − Costo || Gastos de administración y ventas | clase 5 || **Utilidad operacional** | Utilidad bruta − Gastos || Margen bruto % / Margen operacional % | sobre ingresos |Constrúyelo como un DataFrame de dos columnas (`concepto`, `valor`) para poder exportarlo después.

In [ ]:
def saldo_clase(prefijo):    sub = bp[bp["codigo_puc"].str.startswith(prefijo)]    return round(abs(sub["saldo_final"].sum()), 2)ingresos = saldo_clase("41") + saldo_clase("42")costos = saldo_clase("6")gastos = saldo_clase("5")utilidad_bruta = round(ingresos - costos, 2)utilidad_operacional = round(utilidad_bruta - gastos, 2)pyg = pd.DataFrame({    "concepto": ["Ingresos", "Costo de ventas", "Utilidad bruta",                 "Gastos operacionales", "Utilidad operacional",                 "Margen bruto %", "Margen operacional %"],    "valor": [ingresos, costos, utilidad_bruta, gastos, utilidad_operacional,              round(utilidad_bruta / ingresos * 100, 2),              round(utilidad_operacional / ingresos * 100, 2)],})pyg

## Ejercicio 14 — Conciliación bancariaEl ejercicio completo. En libros, el banco es la cuenta `111005`; en el extracto, cada fila trae `referencia` (el comprobante) y `valor`.1. Movimientos de banco en libros con su valor neto (`debito - credito`)2. Movimientos del extracto agrupados por `referencia`3. `merge` con `how="outer"` e `indicator=True`4. Clasifica las partidas conciliatorias:   - **solo en libros** → consignaciones o cheques pendientes   - **solo en extracto** → GMF, comisiones, rendimientos, notas débito   - **en ambos con diferencia de valor** → error de digitación5. Prueba el saldo: `saldo extracto + partidas solo en libros − partidas solo en extracto = saldo en libros`

In [ ]:
libros_banco = (aux[aux["codigo_puc"] == "111005"]                .assign(valor_libros=lambda d: d["debito"] - d["credito"])                .groupby("comprobante")["valor_libros"].sum()                .reset_index()                .rename(columns={"comprobante": "referencia"}))extracto = ext.groupby("referencia")["valor"].sum().reset_index().rename(    columns={"valor": "valor_extracto"})conc = libros_banco.merge(extracto, on="referencia", how="outer", indicator=True)conc[["valor_libros", "valor_extracto"]] = conc[["valor_libros", "valor_extracto"]].fillna(0)conc["diferencia"] = (conc["valor_libros"] - conc["valor_extracto"]).round(2)conc["situacion"] = conc["_merge"].map({    "left_only": "SOLO EN LIBROS (partida en transito)",    "right_only": "SOLO EN EXTRACTO (registrar en libros)",    "both": "CONCILIADA",}).astype(str)   # .map sobre _merge devuelve una categoria; la pasamos a textoconc.loc[(conc["_merge"] == "both") & (conc["diferencia"] != 0), "situacion"] = "DIFERENCIA DE VALOR"print(conc["situacion"].value_counts().to_string(), "\n")solo_libros = conc.loc[conc["_merge"] == "left_only", "valor_libros"].sum()solo_extracto = conc.loc[conc["_merge"] == "right_only", "valor_extracto"].sum()saldo_extracto = ext["valor"].sum() + 185_000_000saldo_libros = (aux.loc[aux["codigo_puc"] == "111005", "debito"].sum()                - aux.loc[aux["codigo_puc"] == "111005", "credito"].sum()                + 185_000_000)print(f"Saldo segun extracto            : {saldo_extracto:>18,.2f}")print(f"(+) Partidas solo en libros     : {solo_libros:>18,.2f}")print(f"(-) Partidas solo en extracto   : {-solo_extracto:>18,.2f}")print(f"(=) Saldo conciliado            : {saldo_extracto + solo_libros - solo_extracto:>18,.2f}")print(f"Saldo segun libros              : {saldo_libros:>18,.2f}")print("CONCILIACION OK" if round(saldo_extracto + solo_libros - solo_extracto - saldo_libros, 2) == 0      else "REVISAR DIFERENCIA")partidas = conc[conc["situacion"] != "CONCILIADA"].drop(columns="_merge")partidas.head(15)

## Ejercicio 15 — Exportar el informe a ExcelTodo el trabajo termina en un entregable. Con `pd.ExcelWriter` escribes varias hojas en un mismo archivo:```pythonwith pd.ExcelWriter("informe.xlsx", engine="openpyxl") as w:    df1.to_excel(w, sheet_name="Hoja1", index=False)    df2.to_excel(w, sheet_name="Hoja2", index=False)```Exporta a `salidas/informe_contable.xlsx` cuatro hojas: balance por clase, estado de resultados, partidas conciliatorias y facturas con alerta.

In [ ]:
import osos.makedirs("salidas", exist_ok=True)with pd.ExcelWriter("salidas/informe_contable.xlsx", engine="openpyxl") as w:    bp.to_excel(w, sheet_name="Balance de prueba", index=False)    pyg.to_excel(w, sheet_name="Estado de resultados", index=False)    partidas.to_excel(w, sheet_name="Conciliacion", index=False)    fac_limpio[fac_limpio["alerta"] == "REVISAR"].to_excel(        w, sheet_name="Facturas con alerta", index=False)print("Archivo generado: salidas/informe_contable.xlsx")print("Hojas: Balance de prueba | Estado de resultados | Conciliacion | Facturas con alerta")

---## Retos para seguir practicando1. **Cartera por edades**: clasifica el saldo de clientes (`130505`) en 0–30, 31–60, 61–90 y +90 días   usando `pd.cut` sobre los días transcurridos desde la fecha del documento.2. **Certificados de retención**: agrupa la retefuente practicada (`236540`) por NIT y por mes,   y genera un archivo por proveedor con `to_excel` dentro de un `for`.3. **Formato 1001 (información exógena)**: arma un DataFrame con NIT, concepto, pago acumulado   y retención practicada por tercero para el trimestre.4. **Indicadores**: calcula razón corriente, capital de trabajo y rotación de inventarios   con los saldos del balance.5. **Gráfico**: `ventas_mes.plot(kind="bar")` — necesitas `pip install matplotlib`.## Chuleta de pandas| Necesito | Código ||---|---|| leer / escribir | `pd.read_csv()` · `pd.read_excel()` · `df.to_excel()` || ver | `.head()` `.tail()` `.info()` `.describe()` `.shape` `.columns` || filtrar | `df[df["col"] > 0]` · `.isin([...])` · `.between(a, b)` · `.query("col > 0")` || texto | `.str.strip()` `.str.upper()` `.str.contains()` `.str.startswith()` || fechas | `pd.to_datetime()` · `.dt.month` `.dt.year` `.dt.to_period("M")` || nulos | `.isna().sum()` · `.fillna(0)` · `.dropna()` || agrupar | `.groupby("col")["val"].sum()` · `.agg(["sum", "count"])` · `.pivot_table()` || cruzar | `.merge(otro, on="llave", how="left/outer", indicator=True)` || ordenar | `.sort_values("col", ascending=False)` · `.nlargest(5, "col")` || nueva columna | `df["nueva"] = ...` · `.assign(nueva=...)` · `np.where(cond, a, b)` |## Si algo falla| Error | Causa habitual ||---|---|| `FileNotFoundError` | VS Code no está parado en la carpeta del proyecto — usa *Abrir carpeta*, no *Abrir archivo* || `NameError` | no ejecutaste la celda anterior || `KeyError: 'columna'` | el nombre no existe: revisa `df.columns` || `SettingWithCopyWarning` | estás modificando un filtro; usa `.copy()` || suma que no cuadra por centavos | falta `.round(2)` antes de comparar |